# Fluency Comparison

Compare two lessons for the same student across all fluency metrics.

**Data sources:**
- `sentences-analysis.json` — pre-computed sentence-level metrics (fluency score, gaps, fillers, duplicates, accuracy)
- `fluidity.json` — word-level timing, speed and filler annotations

Word speed thresholds are computed from **both lessons combined** so that buckets are directly comparable. All sentences are included — no sampling.

In [ ]:
import json, pathlib, html as _html
from collections import defaultdict, Counter
from IPython.display import display, HTML

# ── USER CONFIG ────────────────────────────────────────────────────────────────
STUDENT  = 'Student-2'
LESSON_1 = 'lesson-1'
LESSON_2 = 'lesson-2'
SAMPLE   = 0.15  # fraction of sentences used for this demo

# ── Path resolution ────────────────────────────────────────────────────────────
here = pathlib.Path().resolve()
root = here
for _ in range(4):
    if (root / 'processed_data').exists():
        break
    root = root.parent

def _data_path(kind, lesson):
    filename = 'fluidity.json' if kind == 'fluidity' else 'sentences-analysis.json'
    return root / 'processed_data' / kind / STUDENT / lesson / filename

In [ ]:
def _load_lesson(label):
    sa = json.loads(_data_path('sentences_analysis', label).read_text('utf-8'))
    fl = json.loads(_data_path('fluidity',           label).read_text('utf-8'))
    n  = max(1, int(len(sa['sentences']) * SAMPLE))
    sa['sentences'] = sa['sentences'][:n]
    sw = defaultdict(list)
    for w in fl['words']:
        sw[w['sentence_id']].append(w)
    return sa, fl, sw

sa1, fl1, sw1 = _load_lesson(LESSON_1)
sa2, fl2, sw2 = _load_lesson(LESSON_2)

# Global speed percentile thresholds — both lessons combined, content words only
_combined = sorted(
    w['speed']
    for fl in (fl1, fl2)
    for w in fl['words']
    if w['speed'] is not None and not w['is_filler']
)
_n = len(_combined) or 1
SPEED_P25 = _combined[_n // 4]
SPEED_P75 = _combined[3 * _n // 4]
SPEED_P90 = _combined[9 * _n // 10]

print(f'Loaded {STUDENT} (demo {int(SAMPLE*100)}%): '
      f'{LESSON_1} ({len(sa1["sentences"])} sentences) | '
      f'{LESSON_2} ({len(sa2["sentences"])} sentences)')
print(f'Global thresholds — fast < {round(SPEED_P25*1000)}ms | '
      f'normal {round(SPEED_P25*1000)}-{round(SPEED_P75*1000)}ms | '
      f'slow {round(SPEED_P75*1000)}-{round(SPEED_P90*1000)}ms | '
      f'very slow > {round(SPEED_P90*1000)}ms/letter')

In [ ]:
# ── Colour constants ───────────────────────────────────────────────────────────
FILLER_BG     = '#FFF7ED'
FILLER_BORDER = '#F97316'
FILLER_TEXT   = '#9A3412'
DUP_1_BG = '#FEF2F2'; DUP_1_BD = '#FECACA'; DUP_1_TX = '#DC2626'
DUP_N_BG = '#FEE2E2'; DUP_N_BD = '#FCA5A5'; DUP_N_TX = '#991B1B'
GAP_FAST = 0.08
GAP_MED  = 0.25


def _speed_style(speed):
    if speed is None:
        return 'background:#F9FAFB;border:1px solid #E5E7EB;color:#9CA3AF;'
    if speed < SPEED_P25:
        return 'background:#D1FAE5;border:1px solid #34D399;color:#064E3B;'
    if speed <= SPEED_P75:
        return 'background:#F3F4F6;border:1px solid #D1D5DB;color:#374151;'
    if speed <= SPEED_P90:
        return 'background:#FEF3C7;border:1px solid #FCD34D;color:#78350F;'
    return 'background:#FEE2E2;border:1px solid #FCA5A5;color:#991B1B;'


def _ms(speed):
    return str(round(speed * 1000)) + 'ms' if speed is not None else '\u2014'


def _gap_col(g):
    if g is None or g < 0: return '#D1D5DB'
    if g < GAP_FAST:  return '#34D399'
    if g < GAP_MED:   return '#FCD34D'
    return '#F87171'


def _gap_ms_str(g):
    return (str(round(g * 1000)) + 'ms') if g is not None else '?'


def _chip(word_dict):
    w = word_dict
    label = w['punctuated_word']
    if w['is_filler']:
        chip_s = ('background:' + FILLER_BG + ';border:1px solid ' + FILLER_BORDER + ';'
                  'color:' + FILLER_TEXT + ';font-style:italic;border-radius:6px;'
                  'padding:4px 8px;display:inline-block;text-align:center;')
        sub_s  = 'font-size:9px;color:' + FILLER_BORDER + ';margin-top:2px;text-align:center;'
        sub_t  = w['filler_type'] or 'filler'
    else:
        chip_s = (_speed_style(w['speed'])
                  + 'border-radius:6px;padding:4px 8px;display:inline-block;text-align:center;')
        sub_s  = 'font-size:9px;color:#9CA3AF;margin-top:2px;text-align:center;'
        sub_t  = _ms(w['speed'])
    return ('<div style="display:inline-block;margin:3px;">'
            + '<div style="' + chip_s + 'font-size:13px;font-family:Georgia,serif;white-space:nowrap;">'
            + label + '</div>'
            + '<div style="' + sub_s + '">' + sub_t + '</div>'
            + '</div>')


def _speed_gaps_row(ws):
    items = []
    for i, w in enumerate(ws):
        items.append(_chip(w))
        if i < len(ws) - 1:
            g   = ws[i + 1]['start'] - ws[i]['end']
            col = _gap_col(g)
            gv  = _gap_ms_str(g)
            items.append(
                '<div style="display:inline-block;text-align:center;vertical-align:middle;'
                'margin:0 1px;width:26px;">'
                + '<div style="width:2px;height:18px;background:' + col + ';margin:0 auto;"></div>'
                + '<div style="font-size:8px;color:' + col + ';margin-top:1px;">' + gv + '</div>'
                + '</div>'
            )
    return '<div style="line-height:3.8;padding:2px 0;">' + ''.join(items) + '</div>'


def _make_dup_map(dups):
    dm = {}
    for d in dups:
        k     = len(d['phrase'])
        label = ' '.join(d['phrase'])
        mtype = d['match_type']
        for occ_num, start in enumerate(d['start_indices']):
            for p in range(k):
                dm[start + p] = (occ_num == 0, label, mtype)
    return dm


def _fillers_dups_row(ws, dups):
    dm     = _make_dup_map(dups)
    tokens = []
    for i, w in enumerate(ws):
        word = w['punctuated_word']
        if w.get('is_filler'):
            ft = w.get('filler_type', 'filler')
            s  = ('background:' + FILLER_BG + ';border:1px solid ' + FILLER_BORDER + ';'
                  'color:' + FILLER_TEXT + ';font-style:italic;border-radius:4px;padding:1px 5px;')
            tokens.append('<span style="' + s + '" title="' + ft + '">' + word + '</span>')
        elif i in dm:
            is_first, label, mtype = dm[i]
            fuzz = ' ~' if mtype == 'fuzzy' else ''
            if is_first:
                s = ('background:' + DUP_1_BG + ';color:' + DUP_1_TX + ';'
                     'border-radius:4px;padding:1px 5px;border-bottom:2px dotted ' + DUP_1_TX + ';')
                tokens.append('<span style="' + s + '" title="first of: ' + label + fuzz + '">' + word + '</span>')
            else:
                s = ('background:' + DUP_N_BG + ';color:' + DUP_N_TX + ';'
                     'border-radius:4px;padding:1px 5px;text-decoration:line-through;')
                tokens.append('<span style="' + s + '" title="extra copy: ' + label + fuzz + '">' + word + '</span>')
        else:
            tokens.append(word)
    notes = []
    for d in dups:
        fuzz = '~' if d['match_type'] == 'fuzzy' else ''
        notes.append('\u201c' + ' '.join(d['phrase']) + '\u201d' + fuzz + ' x' + str(d['occurrences']))
    note_html = ''
    if notes:
        note_html = ('<div style="font-size:10px;color:#9CA3AF;margin-top:4px;">'
                     + 'repeats: ' + '  |  '.join(notes) + '</div>')
    return ('<div style="font-family:Georgia,serif;line-height:2.0;font-size:14px;padding:2px 0;">'
            + ' '.join(tokens) + '</div>' + note_html)


def _score_col(score):
    if score is None:  return '#9CA3AF', '#F3F4F6'
    if score >= 80:    return '#059669', '#D1FAE5'
    if score >= 60:    return '#D97706', '#FEF3C7'
    return                    '#DC2626', '#FEE2E2'


def _fluency_panel(fluency_rec):
    score = fluency_rec.get('score')
    comps = fluency_rec.get('components', {})
    if score is None:
        return ''
    fg, bg = _score_col(score)
    comp_labels  = [('speed', 'Speed'), ('gaps', 'Thinking time'),
                    ('fillers', 'Fillers'), ('dups', 'Duplicates')]
    comp_weights = {'speed': 35, 'gaps': 35, 'fillers': 15, 'dups': 15}
    pill = ('<div style="display:inline-block;background:' + bg + ';border:2px solid ' + fg + ';'
            'border-radius:8px;padding:6px 14px;margin-right:20px;text-align:center;vertical-align:top;">'
            + '<div style="font-size:26px;font-weight:800;color:' + fg + ';line-height:1;">'
            + str(score) + '</div>'
            + '<div style="font-size:9px;color:' + fg + ';margin-top:2px;">/ 100</div>'
            + '</div>')
    bars = ''
    for key, label in comp_labels:
        val = comps.get(key, 0)
        w   = comp_weights[key]
        cv, cbg = _score_col(val)
        bars += ('<div style="display:flex;align-items:center;gap:8px;margin-bottom:5px;">'
                 + '<span style="font-size:10px;color:#6B7280;width:90px;">'
                 + label + ' <span style="color:#9CA3AF;">&times;' + str(w) + '%</span></span>'
                 + '<div style="flex:1;background:#F3F4F6;border-radius:3px;height:12px;">'
                 + '<div style="width:' + str(val) + '%;background:' + cv + ';height:12px;border-radius:3px;"></div>'
                 + '</div>'
                 + '<span style="font-size:10px;font-weight:600;color:' + cv + ';width:26px;text-align:right;">'
                 + str(round(val)) + '</span>'
                 + '</div>')
    return ('<div style="display:flex;align-items:flex-start;flex-wrap:wrap;gap:10px;'
            'background:#FAFAFA;border-radius:6px;padding:10px 12px;margin-bottom:10px;">'
            + pill + '<div style="flex:1;min-width:220px;">' + bars + '</div>'
            + '</div>')


def _bdg(text, bg, col, bdr):
    return ('<span style="background:' + bg + ';color:' + col + ';border:1px solid ' + bdr + ';'
            'border-radius:10px;padding:1px 7px;font-size:10px;white-space:nowrap;">'
            + text + '</span>')


def _sentence_gap_ms(ws):
    gaps = [ws[i + 1]['start'] - ws[i]['end']
            for i in range(len(ws) - 1)
            if ws[i + 1]['start'] - ws[i]['end'] >= 0]
    return round(sum(gaps) / len(gaps) * 1000) if gaps else None


def _card(sa_rec, ws):
    dups      = sa_rec.get('duplicates', [])
    n_fillers = sa_rec.get('fillers', {}).get('count', 0)
    n_extra   = sum(d['occurrences'] - 1 for d in dups)
    gap_ms    = _sentence_gap_ms(ws)
    acc_mean  = sa_rec.get('accuracy', {}).get('mean')
    fluency   = sa_rec.get('fluency', {})
    score     = fluency.get('score')
    sid       = sa_rec.get('sentence_id', '')
    preview   = _html.escape(sa_rec.get('text', ''))
    if len(preview) > 72:
        preview = preview[:69] + '...'
    badges = []
    if score is not None:
        fg, bg = _score_col(score)
        badges.append(_bdg('fluency ' + str(score), bg, fg, fg))
    if gap_ms is not None:
        gms = gap_ms / 1000
        if   gms < GAP_FAST: gb, gc, gbd = '#D1FAE5', '#065F46', '#34D399'
        elif gms < GAP_MED:  gb, gc, gbd = '#FEF3C7', '#78350F', '#FCD34D'
        else:                 gb, gc, gbd = '#FEE2E2', '#991B1B', '#FCA5A5'
        badges.append(_bdg('gap ' + str(gap_ms) + 'ms', gb, gc, gbd))
    if n_fillers:
        badges.append(_bdg(str(n_fillers) + ' filler' + ('' if n_fillers == 1 else 's'),
                           FILLER_BG, FILLER_TEXT, FILLER_BORDER))
    else:
        badges.append(_bdg('no fillers', '#F9FAFB', '#9CA3AF', '#E5E7EB'))
    if n_extra:
        badges.append(_bdg(str(n_extra) + ' repeat' + ('' if n_extra == 1 else 's'),
                           DUP_N_BG, DUP_N_TX, DUP_N_BD))
    else:
        badges.append(_bdg('no repeats', '#F9FAFB', '#9CA3AF', '#E5E7EB'))
    if acc_mean is not None:
        pct = round(acc_mean * 100)
        if   acc_mean >= 0.90: ab, ac, abd = '#D1FAE5', '#065F46', '#34D399'
        elif acc_mean >= 0.75: ab, ac, abd = '#FEF3C7', '#78350F', '#FCD34D'
        else:                   ab, ac, abd = '#FEE2E2', '#991B1B', '#FCA5A5'
        badges.append(_bdg('conf ' + str(pct) + '%', ab, ac, abd))
    badge_row = '&nbsp;'.join(badges)
    summary = (
        '<summary style="display:flex;align-items:center;flex-wrap:wrap;gap:6px;'
        'padding:8px 14px;cursor:pointer;background:#FAFAFA;list-style:none;">'
        + '<span style="font-size:10px;color:#9CA3AF;min-width:30px;flex-shrink:0;">s'
        + str(sid) + '</span>'
        + '<span style="font-family:Georgia,serif;font-size:13px;color:#374151;'
        'flex:1;min-width:160px;">' + preview + '</span>'
        + '<span style="font-size:11px;color:#9CA3AF;flex-shrink:0;margin-right:4px;">'
        + str(len(ws)) + 'w&nbsp;&nbsp;</span>'
        + badge_row + '</summary>'
    )
    body = (
        '<div style="padding:12px 16px 14px;border-top:1px solid #E5E7EB;">'
        + _fluency_panel(fluency)
        + '<div style="font-size:10px;font-weight:700;color:#B0B7C0;letter-spacing:.07em;margin-bottom:4px;">'
        + 'SPEED &amp; INTER-WORD GAPS</div>'
        + _speed_gaps_row(ws)
        + '<div style="font-size:10px;font-weight:700;color:#B0B7C0;letter-spacing:.07em;margin:12px 0 4px;">'
        + 'FILLERS &amp; REPETITIONS</div>'
        + _fillers_dups_row(ws, dups)
        + '</div>'
    )
    return ('<details style="border:1px solid #E5E7EB;border-radius:8px;margin-bottom:6px;overflow:hidden;">'
            + summary + body + '</details>')


def compute_lesson_stats(sa_data, fl_data, sent_words):
    sentences = sa_data['sentences']
    words_all = fl_data['words']
    total_words  = len(words_all)
    filler_count = sum(1 for w in words_all if w['is_filler'])
    filler_rate  = round(filler_count / total_words * 100, 1) if total_words else 0.0
    fluency_scores = [s['fluency']['score'] for s in sentences
                      if s.get('fluency', {}).get('score') is not None]
    avg_fluency    = round(sum(fluency_scores) / len(fluency_scores), 1) if fluency_scores else 0.0
    acc_vals = [s['accuracy']['mean'] for s in sentences
                if s.get('accuracy', {}).get('mean') is not None]
    avg_acc  = round(sum(acc_vals) / len(acc_vals) * 100, 1) if acc_vals else 0.0
    all_gaps = []
    for sid, ws in sent_words.items():
        for i in range(len(ws) - 1):
            g = ws[i + 1]['start'] - ws[i]['end']
            if g >= 0:
                all_gaps.append(g)
    avg_gap_ms = round(sum(all_gaps) / len(all_gaps) * 1000) if all_gaps else 0
    comp_sums  = {'speed': 0.0, 'gaps': 0.0, 'fillers': 0.0, 'dups': 0.0}
    comp_count = {'speed': 0,   'gaps': 0,   'fillers': 0,   'dups': 0}
    for s in sentences:
        comps = s.get('fluency', {}).get('components', {})
        for k in comp_sums:
            v = comps.get(k)
            if v is not None:
                comp_sums[k]  += v
                comp_count[k] += 1
    comp_avgs = {k: round(comp_sums[k] / comp_count[k], 1) if comp_count[k] else 0.0
                 for k in comp_sums}
    speed_buckets = {'fast': 0, 'normal': 0, 'slow': 0, 'very_slow': 0, 'total': 0}
    for w in words_all:
        sp = w.get('speed')
        if sp is None:
            continue
        speed_buckets['total'] += 1
        if   sp < SPEED_P25:  speed_buckets['fast']      += 1
        elif sp <= SPEED_P75: speed_buckets['normal']    += 1
        elif sp <= SPEED_P90: speed_buckets['slow']      += 1
        else:                  speed_buckets['very_slow'] += 1
    filler_types = Counter(
        w['filler_type'] for w in words_all if w['is_filler'] and w.get('filler_type')
    )
    score_dist = [0, 0, 0, 0, 0]
    for s in sentences:
        sc = s.get('fluency', {}).get('score')
        if sc is not None:
            score_dist[min(int(sc // 20), 4)] += 1
    sentences_sorted = sorted(
        sentences,
        key=lambda s: s.get('fluency', {}).get('score') or 0
    )
    return {
        'sentence_count':   len(sentences),
        'word_count':       total_words,
        'filler_count':     filler_count,
        'filler_rate':      filler_rate,
        'avg_fluency':      avg_fluency,
        'avg_accuracy':     avg_acc,
        'avg_gap_ms':       avg_gap_ms,
        'comp_avgs':        comp_avgs,
        'speed_buckets':    speed_buckets,
        'filler_types':     filler_types,
        'score_dist':       score_dist,
        'sentences_sorted': sentences_sorted,
    }


print('Visual helpers and compute_lesson_stats defined.')

In [ ]:
FTYPE_COLORS = {
    'backchannel':      '#3B82F6',
    'hesitation':       '#8B5CF6',
    'discourse_marker': '#F97316',
    'placeholder':      '#EF4444',
}
FTYPE_FALLBACK = '#6B7280'


def _pct(count, total):
    return round(count / total * 100, 1) if total else 0.0


def _hbar(label, value_pct, count, color, lw='120px'):
    count_str = str(count) if count != '' else ''
    return ('<div style="display:flex;align-items:center;gap:10px;margin-bottom:7px;">'
            + '<span style="width:' + lw + ';font-size:11px;color:#374151;">' + label + '</span>'
            + '<div style="flex:1;background:#E5E7EB;border-radius:3px;height:16px;">'
            + '<div style="width:' + str(value_pct) + '%;background:' + color + ';height:16px;border-radius:3px;"></div>'
            + '</div>'
            + '<span style="width:40px;text-align:right;font-size:11px;font-weight:600;">' + count_str + '</span>'
            + '</div>')


def _stats_cards_html(stats):
    fg_fl = _score_col(stats['avg_fluency'])[0]
    cards = [
        ('Sentences',    str(stats['sentence_count']),           '#374151'),
        ('Words',        str(stats['word_count']),               '#374151'),
        ('Avg fluency',  str(stats['avg_fluency']),              fg_fl),
        ('Filler rate',  str(stats['filler_rate']) + '%',        FILLER_BORDER),
        ('Avg accuracy', str(stats['avg_accuracy']) + '%',       '#059669'),
        ('Avg gap',      str(stats['avg_gap_ms']) + 'ms',        '#6B7280'),
    ]
    html = '<div style="display:flex;flex-wrap:wrap;gap:6px;margin-bottom:10px;">'
    for lbl, val, col in cards:
        html += ('<div style="background:#F9FAFB;border:1px solid #E5E7EB;border-radius:8px;'
                 'padding:8px 12px;text-align:center;flex:1;min-width:70px;">'
                 + '<div style="font-size:16px;font-weight:800;color:' + col + ';">' + val + '</div>'
                 + '<div style="font-size:10px;color:#6B7280;margin-top:2px;">' + lbl + '</div>'
                 + '</div>')
    return html + '</div>'


def _speed_chart_html(stats):
    sb    = stats['speed_buckets']
    total = sb['total']
    rows  = [
        ('Fast',      sb['fast'],      '#34D399'),
        ('Normal',    sb['normal'],    '#9CA3AF'),
        ('Slow',      sb['slow'],      '#FCD34D'),
        ('Very slow', sb['very_slow'], '#FCA5A5'),
    ]
    html = ''
    for lbl, cnt, col in rows:
        p = _pct(cnt, total)
        html += _hbar(lbl + '  ' + str(p) + '%', p, cnt, col)
    html += ('<div style="font-size:9px;color:#9CA3AF;margin-top:4px;">'
             + f'fast &lt; {round(SPEED_P25*1000)}ms'
             + f'  |  slow &gt; {round(SPEED_P75*1000)}ms'
             + f'  |  very slow &gt; {round(SPEED_P90*1000)}ms/letter'
             + '</div>')
    return html


def _filler_type_chart_html(stats):
    ft    = stats['filler_types']
    total = sum(ft.values())
    if not total:
        return '<div style="font-size:12px;color:#9CA3AF;">No fillers detected</div>'
    html = ''
    ordered = ['hesitation', 'backchannel', 'discourse_marker', 'placeholder']
    for ftype in ordered:
        cnt = ft.get(ftype, 0)
        if cnt == 0:
            continue
        p   = _pct(cnt, total)
        col = FTYPE_COLORS.get(ftype, FTYPE_FALLBACK)
        html += _hbar(ftype + '  ' + str(p) + '%', p, cnt, col)
    for ftype, cnt in ft.items():
        if ftype not in ordered:
            p = _pct(cnt, total)
            html += _hbar(ftype + '  ' + str(p) + '%', p, cnt, FTYPE_FALLBACK)
    return html


def _score_dist_chart_html(stats):
    sd     = stats['score_dist']
    labels = ['0\u201320', '20\u201340', '40\u201360', '60\u201380', '80\u2013100']
    mids   = [10, 30, 50, 70, 90]
    total  = sum(sd)
    html   = ''
    for lbl, mid, cnt in zip(labels, mids, sd):
        p   = _pct(cnt, total)
        col = _score_col(mid)[0]
        html += _hbar(lbl + '  ' + str(p) + '%', p, cnt, col)
    return html


def _component_chart_html(stats):
    ca    = stats['comp_avgs']
    items = [
        ('Speed',         'speed',   35),
        ('Thinking time', 'gaps',    35),
        ('Fillers',       'fillers', 15),
        ('Duplicates',    'dups',    15),
    ]
    html = ''
    for label, key, weight in items:
        val   = ca[key]
        color = _score_col(val)[0]
        lbl   = label + ' <span style="color:#9CA3AF;">&times;' + str(weight) + '%</span>'
        html += ('<div style="display:flex;align-items:center;gap:10px;margin-bottom:7px;">'
                 + '<span style="width:150px;font-size:11px;color:#374151;">' + lbl + '</span>'
                 + '<div style="flex:1;background:#E5E7EB;border-radius:3px;height:16px;">'
                 + '<div style="width:' + str(val) + '%;background:' + color + ';height:16px;border-radius:3px;"></div>'
                 + '</div>'
                 + '<span style="width:40px;text-align:right;font-size:11px;font-weight:600;color:' + color + ';">' + str(val) + '</span>'
                 + '</div>')
    return html


print('Chart helpers defined.')

In [ ]:
stats1 = compute_lesson_stats(sa1, fl1, sw1)
stats2 = compute_lesson_stats(sa2, fl2, sw2)


def _sent_list_html(stats, sent_words_dict):
    parts = []
    for sa_rec in stats['sentences_sorted']:
        sid = sa_rec.get('sentence_id')
        ws  = sent_words_dict.get(sid, [])
        parts.append(_card(sa_rec, ws))
    return ''.join(parts)


def _sec(title):
    return ('<div style="font-size:.76em;font-weight:600;color:#607d8b;'
            'letter-spacing:.04em;text-transform:uppercase;margin:16px 0 5px;">'
            + title + '</div>')


def _card_box(inner):
    return ('<div style="background:#F9FAFB;border:1px solid #E5E7EB;'
            'border-radius:6px;padding:12px 14px;margin-bottom:4px;">'
            + inner + '</div>')


def _col_html(lesson_label, stats, sent_words_dict, list_id, head_color):
    content = (
        _sec('Aggregate Stats')
        + _stats_cards_html(stats)
        + _sec('Word Speed Distribution')
        + _card_box(_speed_chart_html(stats))
        + _sec('Filler Type Breakdown')
        + _card_box(_filler_type_chart_html(stats))
        + _sec('Fluency Score Distribution')
        + _card_box(_score_dist_chart_html(stats))
        + _sec('Component Score Averages')
        + _card_box(_component_chart_html(stats))
        + _sec('Per-Sentence Detail \u2014 sorted worst \u2192 best  (click to expand)')
        + '<div id="' + list_id + '" style="height:520px;overflow-y:auto;'
        'border:1px solid #E5E7EB;border-radius:6px;background:#fff;padding:6px;">'
        + _sent_list_html(stats, sent_words_dict)
        + '</div>'
    )
    return (
        '<div>'
        + '<div style="font-size:1em;font-weight:700;padding:6px 14px;'
        'border-radius:6px 6px 0 0;color:#fff;background:' + head_color + ';">' + lesson_label + '</div>'
        + '<div style="border:1px solid #cfd8dc;border-radius:0 0 6px 6px;'
        'border-top:none;padding:12px;background:#fff;">' + content + '</div>'
        + '</div>'
    )


CSS = '''
<style>
  .fcp-root { font-family: sans-serif; }
  details > summary { list-style: none; }
  details > summary::-webkit-details-marker { display: none; }
  details > summary:hover { background: #F0F9FF !important; }
</style>
'''

JS = '''
<script>
(function() {
  var l1 = document.getElementById('fcp-list-l1');
  var l2 = document.getElementById('fcp-list-l2');
  if (!l1 || !l2) return;
  var lk = false;
  l1.addEventListener('scroll', function() { if (lk) return; lk = true; l2.scrollTop = l1.scrollTop; lk = false; });
  l2.addEventListener('scroll', function() { if (lk) return; lk = true; l1.scrollTop = l2.scrollTop; lk = false; });
})();
</script>
'''

html_out = (
    CSS
    + '<div class="fcp-root">'
    + '<div style="font-size:1.3em;font-weight:700;margin-bottom:14px;color:#263238;">'
    + _html.escape(STUDENT) + ' \u2014 Fluency Comparison: '
    + _html.escape(LESSON_1) + ' vs ' + _html.escape(LESSON_2)
    + '</div>'
    + '<div style="display:grid;grid-template-columns:1fr 1fr;gap:20px;">'
    + _col_html(LESSON_1, stats1, sw1, 'fcp-list-l1', '#1565c0')
    + _col_html(LESSON_2, stats2, sw2, 'fcp-list-l2', '#2e7d32')
    + '</div>'
    + '</div>'
    + JS
)

display(HTML(html_out))